# CatBoost Regression

## Objective

Build a CatBoost regression model for Ford car price prediction.

## Goals

- Understand CatBoost
- Handle categorical features
- Build a baseline CatBoost model
- Evaluate model performance
- Tune hyperparameters
- Compare CatBoost with XGBoost, Random Forest, and Gradient Boosting

In [1]:
from catboost import CatBoostRegressor 

import pandas as pd 
import numpy as np

from sklearn.model_selection import train_test_split 
import joblib

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
df = pd.read_csv("../data/interim/featured_ford.csv")

In [3]:
X = df.drop(columns=["price" , "price_segment"])
y = df["price"] 

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [5]:
X_train.dtypes

model                   str
year                  int64
transmission            str
mileage               int64
fuelType                str
tax                   int64
mpg                 float64
engineSize          float64
car_age               int64
mileage_per_year    float64
engine_category         str
fuel_efficiency         str
tax_category            str
dtype: object

In [6]:
categorical_features = X_train.select_dtypes(
    include=["object","category"]
).columns.tolist()

categorical_features

C:\Users\shutt\AppData\Local\Temp\ipykernel_2880\4037794552.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


['model',
 'transmission',
 'fuelType',
 'engine_category',
 'fuel_efficiency',
 'tax_category']

In [7]:
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

numerical_features 

['year', 'mileage', 'tax', 'mpg', 'engineSize', 'car_age', 'mileage_per_year']

In [8]:
cat_features = categorical_features 

In [9]:
cat_model = CatBoostRegressor(
    iterations = 500,
    learning_rate = 0.05,
    depth = 6,
    loss_function = "RMSE",
    random_seed = 42 ,
    verbose = False 

)

In [10]:
cat_model.fit(
    X_train , 
    y_train,
    cat_features = categorical_features 
)

CatBoostRegressor(depth=6, iterations=500, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=False)

In [11]:
cat_predictions = cat_model.predict(X_test)

In [12]:
cat_mae = mean_absolute_error(y_test, cat_predictions)

cat_mse = mean_squared_error(y_test, cat_predictions)

cat_rmse = np.sqrt(cat_mse)

cat_r2 = r2_score(y_test, cat_predictions)

print("CatBoost Performance")
print("=" * 40)
print(f"MAE  : {cat_mae:.2f}")
print(f"MSE  : {cat_mse:.2f}")
print(f"RMSE : {cat_rmse:.2f}")
print(f"R2   : {cat_r2:.4f}")
print("=" * 40)

CatBoost Performance
MAE  : 854.47
MSE  : 1637482.84
RMSE : 1279.64
R2   : 0.9278


In [13]:
param_distributions_cat = {
    "iterations": [300, 500, 700, 1000],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "depth": [4, 5, 6, 7, 8, 10],
    "l2_leaf_reg": [1, 3, 5, 10],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "random_strength": [0, 0.5, 1, 2]
}

In [14]:
cat_model_tuning = CatBoostRegressor(
    loss_function="RMSE",
    random_seed=42,
    verbose = False
)

In [15]:
from sklearn.model_selection import RandomizedSearchCV

In [16]:
cat_search = RandomizedSearchCV(
    estimator= cat_model_tuning,
    param_distributions= param_distributions_cat,
    n_iter=30,
    cv = 5,
    scoring= "neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [17]:
cat_search.fit(X_train, y_train , cat_features = categorical_features)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


KeyboardInterrupt: 

In [22]:
cat_search.best_params_

{'subsample': 0.8,
 'random_strength': 1,
 'learning_rate': 0.1,
 'l2_leaf_reg': 1,
 'iterations': 500,
 'depth': 7}

In [ ]:
-cat_search.best_score_

np.float64(1149.043820018313)

In [26]:
best_cat_model = cat_search.best_estimator_

In [29]:
tuned_cat_predictions = best_cat_model.predict(X_test)

In [30]:
tuned_cat_mae = mean_absolute_error(
    y_test,
    tuned_cat_predictions
)

tuned_cat_mse = mean_squared_error(
    y_test,
    tuned_cat_predictions
)

tuned_cat_rmse = np.sqrt(tuned_cat_mse)

tuned_cat_r2 = r2_score(
    y_test,
    tuned_cat_predictions
)

print("Tuned CatBoost Performance")
print("=" * 40)
print(f"MAE  : {tuned_cat_mae:.2f}")
print(f"MSE  : {tuned_cat_mse:.2f}")
print(f"RMSE : {tuned_cat_rmse:.2f}")
print(f"R2   : {tuned_cat_r2:.4f}")
print("=" * 40)

Tuned CatBoost Performance
MAE  : 813.01
MSE  : 1509258.80
RMSE : 1228.52
R2   : 0.9334
